<a href="https://colab.research.google.com/github/ElizabethTeena/Data_Visualization/blob/main/Objective2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Installing Prophet
!pip -q install prophet pyarrow

In [ ]:
#Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#To confirm the files exist
import os, glob

BASE = "/content/drive/MyDrive/O2_Files"

LOAD_DIR = f"{BASE}/hrl_load_estimated(2020-2025)"
GEN_DIR  = f"{BASE}/Generation_FuelType(2020-2025)"
NET_DIR  = f"{BASE}/NetImport(2020-2025)"
PEAK_CSV = f"{BASE}/O2_Load_AnnualPeak_PSEG.csv"

print("Base contents:", os.listdir(BASE))

load_files = sorted(glob.glob(f"{LOAD_DIR}/*.csv"))
gen_files  = sorted(glob.glob(f"{GEN_DIR}/*.csv"))
net_files  = sorted(glob.glob(f"{NET_DIR}/*.csv"))

print("Load files:", len(load_files))
print("Gen files:", len(gen_files))
print("Net files:", len(net_files))
print("Peak file exists:", os.path.exists(PEAK_CSV))


Base contents: ['O2_Load_AnnualPeak_PSEG.csv', 'Generation_FuelType(2020-2025)', 'hrl_load_estimated(2020-2025)', 'NetImport(2020-2025)']
Load files: 6
Gen files: 6
Net files: 6
Peak file exists: True


In [ ]:
# Read and combine all files

import pandas as pd

# ----------------------
# LOAD (Hourly)
# ----------------------
load = pd.concat([pd.read_csv(f) for f in load_files], ignore_index=True)

# Parse datetime
load["dt"] = pd.to_datetime(load["datetime_beginning_ept"], errors="coerce")

# Keep PJME (since dataset only contains PJME)
if "load_area" in load.columns:
    load = load[load["load_area"] == "PJME"].copy()

load = load[["dt", "estimated_load_hourly"]] \
        .rename(columns={"estimated_load_hourly": "load_mw"}) \
        .dropna() \
        .sort_values("dt")

print("LOAD rows:", len(load))
print("LOAD range (MW):", load["load_mw"].min(), "to", load["load_mw"].max())


# ----------------------
# GENERATION BY FUEL (Hourly)
# ----------------------
gen = pd.concat([pd.read_csv(f) for f in gen_files], ignore_index=True)

gen["dt"] = pd.to_datetime(
    gen["datetime_beginning_ept"].astype(str).str.strip(),
    format="mixed",
    errors="coerce"
)

# IMPORTANT:
# The generation dataset has NO region column.
# Therefore, we treat it as same PJME proxy region for consistency.

gen = gen[["dt", "fuel_type", "mw", "is_renewable"]] \
        .dropna() \
        .sort_values("dt")

print("GEN rows:", len(gen))
print("GEN range (MW):", gen["mw"].min(), "to", gen["mw"].max())
print("NOTE: Generation treated as same PJME proxy region as load.")


# ----------------------
# NET IMPORTS (Hourly)
# ----------------------
net = pd.concat([pd.read_csv(f) for f in net_files], ignore_index=True)

net["dt"] = pd.to_datetime(net["datetime_beginning_ept"], errors="coerce")

# If state exists, keep NJ (but this may not align perfectly with PJME)
if "state" in net.columns:
    net = net[net["state"] == "NJ"].copy()

net = net[["dt", "net_interchange"]] \
        .rename(columns={"net_interchange": "net_mw"}) \
        .dropna() \
        .sort_values("dt")

print("NET rows:", len(net))
print("NET range (MW):", net["net_mw"].min(), "to", net["net_mw"].max())


# ----------------------
# ANNUAL PEAK
# ----------------------
peak = pd.read_csv(PEAK_CSV)
print("Annual peak columns:", peak.columns.tolist())

print("\nCELL 4 COMPLETED SUCCESSFULLY")


LOAD rows: 52470
LOAD range (MW): 18518 to 58036
GEN rows: 551912
GEN range (MW): 0 to 75926
NOTE: Generation treated as same PJME proxy region as load.
NET rows: 52468
NET range (MW): -9470.99 to 1145.62
Annual peak columns: ['year', 'datetime_beginning_utc', 'datetime_beginning_ept', 'datetime_ending_utc', 'datetime_ending_ept', 'zone', 'nspl_mw']

CELL 4 COMPLETED SUCCESSFULLY


In [ ]:
#checking everything is loaded correctly
print("Gen date range:", gen["dt"].min(), "to", gen["dt"].max())
print("Load date range:", load["dt"].min(), "to", load["dt"].max())
print("Net date range:", net["dt"].min(), "to", net["dt"].max())


Gen date range: 2020-01-01 00:00:00 to 2025-12-31 00:00:00
Load date range: 2020-01-01 00:00:00 to 2025-12-31 00:00:00
Net date range: 2020-01-01 00:00:00 to 2025-12-31 00:00:00


In [ ]:
#  Build daily demand (daily average MW) from hourly load

load_raw = load.copy()  # keep raw safe

daily_demand = (load_raw
                .set_index("dt")["load_mw"]
                .resample("D")
                .mean()
                .reset_index()
                .rename(columns={"dt": "ds", "load_mw": "demand_mw"}))

print("Daily demand rows:", len(daily_demand))
print("Daily demand date range:", daily_demand["ds"].min(), "to", daily_demand["ds"].max())
daily_demand.head()


Daily demand rows: 2192
Daily demand date range: 2020-01-01 00:00:00 to 2025-12-31 00:00:00


,ds,demand_mw
0,2020-01-01,28819.166667
1,2020-01-02,31751.708333
2,2020-01-03,29640.208333
3,2020-01-04,27190.291667
4,2020-01-05,29467.625000


In [ ]:
#Daily Supply calculation from NJ Generation and Net Imports
#   supply_mw = (total NJ generation MW) + (net imports MW)

gen_raw = gen.copy()
net_raw = net.copy()

# Total generation across all fuel types per hour
gen_hourly_total = (gen_raw
                    .groupby("dt", as_index=False)["mw"]
                    .sum()
                    .rename(columns={"mw": "gen_mw"}))

# Convert hourly gen to daily average MW
gen_daily = (gen_hourly_total
             .set_index("dt")["gen_mw"]
             .resample("D")
             .mean()
             .reset_index()
             .rename(columns={"dt": "ds"}))

# Convert net imports to daily average MW
net_daily = (net_raw
             .set_index("dt")["net_mw"]
             .resample("D")
             .mean()
             .reset_index()
             .rename(columns={"dt": "ds"}))

# Merge supply pieces
daily_supply = gen_daily.merge(net_daily, on="ds", how="inner")
daily_supply["supply_mw"] = daily_supply["gen_mw"] + daily_supply["net_mw"]

print("Daily supply rows:", len(daily_supply))
print("Daily supply date range:", daily_supply["ds"].min(), "to", daily_supply["ds"].max())
daily_supply.head()


Daily supply rows: 2192
Daily supply date range: 2020-01-01 00:00:00 to 2025-12-31 00:00:00


,ds,gen_mw,net_mw,supply_mw
0,2020-01-01,86883.833333,-2971.39250,83912.440833
1,2020-01-02,94016.666667,-2655.80750,91360.859167
2,2020-01-03,89775.000000,-2035.38125,87739.618750
3,2020-01-04,85649.833333,-2513.95500,83135.878333
4,2020-01-05,90899.541667,-2796.55500,88102.986667


In [ ]:
#Combining into 1 table and computing gap (Merge demand + supply on date and compute gap)

daily = daily_demand.merge(daily_supply[["ds","gen_mw","net_mw","supply_mw"]], on="ds", how="inner")

daily["gap_mw"] = daily["supply_mw"] - daily["demand_mw"]  # positive = surplus, negative = shortfall

print("Combined daily rows:", len(daily))
print("Combined date range:", daily["ds"].min(), "to", daily["ds"].max())
daily.head()


Combined daily rows: 2192
Combined date range: 2020-01-01 00:00:00 to 2025-12-31 00:00:00


,ds,demand_mw,gen_mw,net_mw,supply_mw,gap_mw
0,2020-01-01,28819.166667,86883.833333,-2971.39250,83912.440833,55093.274167
1,2020-01-02,31751.708333,94016.666667,-2655.80750,91360.859167,59609.150833
2,2020-01-03,29640.208333,89775.000000,-2035.38125,87739.618750,58099.410417
3,2020-01-04,27190.291667,85649.833333,-2513.95500,83135.878333,55945.586667
4,2020-01-05,29467.625000,90899.541667,-2796.55500,88102.986667,58635.361667


In [ ]:
print("Sanity check (daily):")
print(daily[["demand_mw","supply_mw","gap_mw"]].describe())

print("\nTop 5 largest daily surpluses (MW):")
print(daily.sort_values("gap_mw", ascending=False).head(5)[["ds","demand_mw","supply_mw","gap_mw"]])

print("\nTop 5 worst daily shortfalls (MW):")
print(daily.sort_values("gap_mw").head(5)[["ds","demand_mw","supply_mw","gap_mw"]])


Sanity check (daily):
          demand_mw      supply_mw        gap_mw
count   2192.000000    2192.000000   2192.000000
mean   30162.188424   92167.485302  62005.296878
std     4896.122848   12438.194288   7972.110627
min    21457.916667   68403.308750  45574.660417
25%    26272.781250   82065.984896  55660.996354
50%    29200.416667   90504.968542  61202.158333
75%    33260.854167  100144.030938  67164.579271
max    47456.666667  136200.720000  93318.178333

Top 5 largest daily surpluses (MW):
             ds     demand_mw      supply_mw        gap_mw
1847 2025-01-21  42882.541667  136200.720000  93318.178333
1848 2025-01-22  43398.583333  135082.233333  91683.650000
1477 2024-01-17  39802.166667  127211.674583  87409.507917
2031 2025-07-24  38048.875000  124938.372500  86889.497500
2001 2025-06-24  47456.666667  133618.316667  86161.650000

Top 5 worst daily shortfalls (MW):
            ds     demand_mw     supply_mw        gap_mw
515 2021-05-30  23574.875000  69149.535417  45574.660

In [ ]:
!pip install plotly


In [ ]:
#Interactive Demand Plot
import plotly.express as px

fig1 = px.line(
    daily,
    x="ds",
    y="demand_mw",
    title="Daily Electricity Demand (MW)",
    labels={"ds": "Date", "demand_mw": "Demand (MW)"}
)
fig1.update_traces(
    hovertemplate="Date: %{x|%Y-%m-%d}<br>Demand (MW): %{y:,.0f}<extra></extra>"
)

fig1.update_layout(
    hovermode="x unified",
    template="plotly_white"
)

fig1.show()


In [ ]:
#Supply Plot
fig2 = px.line(daily, x="ds", y="supply_mw", title="Daily Supply Proxy (MW)")
fig2.update_traces(
    hovertemplate="Date: %{x|%Y-%m-%d}<br>Supply (MW): %{y:,.0f}<extra></extra>"
)
fig2.update_layout(hovermode="x unified", template="plotly_white")
fig2.show()


In [ ]:
#Gap Plot
fig3 = px.line(daily, x="ds", y="gap_mw", title="Daily Supply–Demand Gap (MW)")
fig3.update_traces(
    hovertemplate="Date: %{x|%Y-%m-%d}<br>Gap (MW): %{y:,.0f}<extra></extra>"
)
fig3.add_hline(y=0)
fig3.update_layout(hovermode="x unified", template="plotly_white")
fig3.show()


In [ ]:
#Some terms to understand in prophet ds-date, yhat_lower -Demand could be at least this much, yhat upper- Demand could be at most this much, yhat - Demand Forecasted

In [ ]:
#Forecast Future Demand Using Prophet


from prophet import Prophet

# Prepare demand data
demand_prophet = daily[["ds", "demand_mw"]].rename(columns={"demand_mw": "y"}).copy()

# Initialize Prophet
m_demand = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)

# Fit model
m_demand.fit(demand_prophet)

# Create future dates (5 years)
future_demand = m_demand.make_future_dataframe(periods=365*5, freq="D")

# Forecast
forecast_demand = m_demand.predict(future_demand)

forecast_demand.head()


,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,weekly,weekly_lower,weekly_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2020-01-01,29114.368870,27426.065666,34257.907077,29114.368870,29114.368870,1733.562062,1733.562062,1733.562062,854.091375,854.091375,854.091375,879.470687,879.470687,879.470687,0.0,0.0,0.0,30847.930932
1,2020-01-02,29117.699569,27475.056615,34287.643540,29117.699569,29117.699569,1655.490488,1655.490488,1655.490488,688.060766,688.060766,688.060766,967.429723,967.429723,967.429723,0.0,0.0,0.0,30773.190058
2,2020-01-03,29121.030268,26874.492381,34029.705760,29121.030268,29121.030268,1288.770855,1288.770855,1288.770855,211.220617,211.220617,211.220617,1077.550238,1077.550238,1077.550238,0.0,0.0,0.0,30409.801123
3,2020-01-04,29124.360967,25280.275994,32222.347994,29124.360967,29124.360967,-265.042960,-265.042960,-265.042960,-1473.382989,-1473.382989,-1473.382989,1208.340029,1208.340029,1208.340029,0.0,0.0,0.0,28859.318007
4,2020-01-05,29127.691666,25254.928437,32568.467902,29127.691666,29127.691666,-390.413605,-390.413605,-390.413605,-1748.211977,-1748.211977,-1748.211977,1357.798373,1357.798373,1357.798373,0.0,0.0,0.0,28737.278061


In [ ]:
#Plotting Demand Forecast
import plotly.express as px

fig_d = px.line(
    forecast_demand,
    x="ds",
    y="yhat",
    title="Forecasted Electricity Demand (MW)",
    labels={"ds": "Date", "yhat": "Demand Forecast (MW)"}
)

fig_d.update_traces(
    hovertemplate="Date: %{x|%Y-%m-%d}<br>Forecast (MW): %{y:,.0f}<extra></extra>"
)

fig_d.update_layout(template="plotly_white")
fig_d.show()


In [ ]:
#Doing Supply Forecast


from prophet import Prophet

# Prepare supply data for Prophet
supply_prophet = daily[["ds", "supply_mw"]].rename(columns={"supply_mw": "y"}).copy()

# Initialize Prophet model
m_supply = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)

# Fit model
m_supply.fit(supply_prophet)

# Create future dates (5 years)
future_supply = m_supply.make_future_dataframe(periods=365*5, freq="D")

# Forecast
forecast_supply = m_supply.predict(future_supply)

# Keep only important columns
supply_forecast_clean = forecast_supply[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()

supply_forecast_clean.head()


,ds,yhat,yhat_lower,yhat_upper
0,2020-01-01,90856.969137,82610.931813,98855.103943
1,2020-01-02,90716.618942,82301.642757,99205.931189
2,2020-01-03,90036.360239,81460.740492,97894.021766
3,2020-01-04,85591.407726,77040.932947,94283.027864
4,2020-01-05,85288.771798,77494.980874,93513.848652


In [ ]:
#Keeping only future supply
last_date = daily["ds"].max()

future_supply_only = supply_forecast_clean[supply_forecast_clean["ds"] > last_date].copy()

future_supply_only.head()


,ds,yhat,yhat_lower,yhat_upper
2192,2026-01-01,99928.423872,91897.025913,108294.577446
2193,2026-01-02,99208.621724,90973.461624,107326.447025
2194,2026-01-03,94728.148582,86933.017774,103193.274556
2195,2026-01-04,94394.737374,86541.555200,102733.383802
2196,2026-01-05,101667.959549,93762.436265,109133.954628


In [ ]:
#Keeping only future demand
# Keep only the columns we need from Prophet output
demand_forecast_clean = forecast_demand[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
# Identify last historical date from your real dataset
last_date = daily["ds"].max()

# Filter only future forecast rows
future_demand_only = demand_forecast_clean[demand_forecast_clean["ds"] > last_date].copy()



future_demand_only.head()


,ds,yhat,yhat_lower,yhat_upper
2192,2026-01-01,32046.628427,28468.696438,35601.577460
2193,2026-01-02,31669.443609,28218.312821,35153.248138
2194,2026-01-03,30105.977008,26509.857318,33399.756762
2195,2026-01-04,29972.014579,26477.087503,33389.947861
2196,2026-01-05,32478.806202,29288.586546,36328.441273


In [ ]:
# Create Low / Medium / High demand scenarios (future only)
# Rename baseline demand and baseline supply for clarity
future_demand_base = future_demand_only[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
future_demand_base = future_demand_base.rename(columns={"yhat": "demand_baseline_mw"})

future_supply_base = future_supply_only[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
future_supply_base = future_supply_base.rename(columns={"yhat": "supply_baseline_mw"})

#  Scenario growth assumptions (CHANGE THESE 3 NUMBERS if needed)
LOW_RATE  = 0.005   # 0.5% per year
MED_RATE  = 0.015   # 1.5% per year
HIGH_RATE = 0.030   # 3.0% per year

#  Apply scenario multipliers to baseline demand
start = future_demand_base["ds"].min()
future_demand_base["t_years"] = (future_demand_base["ds"] - start).dt.days / 365.25

def apply_growth(baseline, t_years, annual_rate):
    return baseline * (1 + annual_rate) ** t_years

future_demand_base["demand_low_mw"]  = apply_growth(future_demand_base["demand_baseline_mw"], future_demand_base["t_years"], LOW_RATE)
future_demand_base["demand_med_mw"]  = apply_growth(future_demand_base["demand_baseline_mw"], future_demand_base["t_years"], MED_RATE)
future_demand_base["demand_high_mw"] = apply_growth(future_demand_base["demand_baseline_mw"], future_demand_base["t_years"], HIGH_RATE)

future_demand_base.head()


,ds,demand_baseline_mw,yhat_lower,yhat_upper,t_years,demand_low_mw,demand_med_mw,demand_high_mw
2192,2026-01-01,32046.628427,28468.696438,35601.577460,0.000000,32046.628427,32046.628427,32046.628427
2193,2026-01-02,31669.443609,28218.312821,35153.248138,0.002738,31669.876063,31670.734571,31672.006645
2194,2026-01-03,30105.977008,26509.857318,33399.756762,0.005476,30106.799222,30108.431516,30110.850210
2195,2026-01-04,29972.014579,26477.087503,33389.947861,0.008214,29973.242421,29975.680033,29979.292151
2196,2026-01-05,32478.806202,29288.586546,36328.441273,0.010951,32480.580261,32484.102342,32489.321630


In [ ]:
#  Compute future Supply–Demand Gap for LOW / MED / HIGH scenarios

gap_scen = future_demand_base.merge(
    future_supply_base[["ds", "supply_baseline_mw"]],
    on="ds",
    how="inner"
)

# Gaps per scenario (Supply - Demand)
gap_scen["gap_low_mw"]  = gap_scen["supply_baseline_mw"] - gap_scen["demand_low_mw"]
gap_scen["gap_med_mw"]  = gap_scen["supply_baseline_mw"] - gap_scen["demand_med_mw"]
gap_scen["gap_high_mw"] = gap_scen["supply_baseline_mw"] - gap_scen["demand_high_mw"]

gap_scen[["ds", "supply_baseline_mw", "gap_low_mw", "gap_med_mw", "gap_high_mw"]].head()


,ds,supply_baseline_mw,gap_low_mw,gap_med_mw,gap_high_mw
0,2026-01-01,99928.423872,67881.795444,67881.795444,67881.795444
1,2026-01-02,99208.621724,67538.745661,67537.887153,67536.615079
2,2026-01-03,94728.148582,64621.349360,64619.717067,64617.298372
3,2026-01-04,94394.737374,64421.494953,64419.057341,64415.445223
4,2026-01-05,101667.959549,69187.379288,69183.857207,69178.637920


In [ ]:
# Plot Future Gaps (Low / Medium / High) — interactive

import plotly.express as px

plot_df = gap_scen.melt(
    id_vars="ds",
    value_vars=["gap_low_mw", "gap_med_mw", "gap_high_mw"],
    var_name="scenario",
    value_name="gap_mw"
)

fig_gap = px.line(
    plot_df,
    x="ds",
    y="gap_mw",
    color="scenario",
    title="Future Supply–Demand Gap (Low / Medium / High Demand Scenarios)",
    labels={"ds": "Date", "gap_mw": "Gap (MW)", "scenario": "Scenario"}
)

fig_gap.add_hline(y=0)

fig_gap.update_traces(
    hovertemplate="Date: %{x|%Y-%m-%d}<br>Gap (MW): %{y:,.0f}<extra></extra>"
)

fig_gap.update_layout(template="plotly_white", hovermode="x unified")
fig_gap.show()


In [ ]:
#Computing Monthly Average
import pandas as pd
import plotly.express as px

gap_month = gap_scen.copy()
gap_month["month"] = gap_month["ds"].dt.to_period("M").dt.to_timestamp()

gap_monthly = (gap_month
               .groupby("month", as_index=False)[["gap_low_mw","gap_med_mw","gap_high_mw"]]
               .min()
               .rename(columns={"month":"ds"}))
gap_monthly.head()


,ds,gap_low_mw,gap_med_mw,gap_high_mw
0,2026-01-01,64421.494953,64419.057341,64415.445223
1,2026-02-01,64398.681431,64352.981888,64285.137687
2,2026-03-01,58587.261700,58531.974732,58449.836552
3,2026-04-01,53800.060045,53730.629102,53627.380431
4,2026-05-01,55387.344558,55308.448476,55191.069206


In [ ]:
#Plotting MonthlyScenario Gap for a cleaner Version to avoid noise
plot_df_m = gap_monthly.melt(
    id_vars="ds",
    value_vars=["gap_low_mw","gap_med_mw","gap_high_mw"],
    var_name="scenario",
    value_name="gap_mw"
)

fig_gap_m = px.line(
    plot_df_m,
    x="ds",
    y="gap_mw",
    color="scenario",
    title="Future Supply–Demand Gap (Monthly Avg) — Low/Med/High Demand Scenarios",
    labels={"ds":"Month", "gap_mw":"Gap (MW)", "scenario":"Scenario"}
)

fig_gap_m.add_hline(y=0)

fig_gap_m.update_traces(
    hovertemplate="Month: %{x|%Y-%m}<br>Gap (MW): %{y:,.0f}<extra></extra>"
)

fig_gap_m.update_layout(template="plotly_white", hovermode="x unified")
fig_gap_m.show()


In [ ]:
#Keeping only rows that has positive surplus
#If gap<0 - no capacity for data center
#If gap>0 -surplus available
dc_capacity = gap_monthly.copy()

dc_capacity["surplus_low"]  = dc_capacity["gap_low_mw"].clip(lower=0)
dc_capacity["surplus_med"]  = dc_capacity["gap_med_mw"].clip(lower=0)
dc_capacity["surplus_high"] = dc_capacity["gap_high_mw"].clip(lower=0)

dc_capacity.head()


,ds,gap_low_mw,gap_med_mw,gap_high_mw,surplus_low,surplus_med,surplus_high
0,2026-01-01,64421.494953,64419.057341,64415.445223,64421.494953,64419.057341,64415.445223
1,2026-02-01,64398.681431,64352.981888,64285.137687,64398.681431,64352.981888,64285.137687
2,2026-03-01,58587.261700,58531.974732,58449.836552,58587.261700,58531.974732,58449.836552
3,2026-04-01,53800.060045,53730.629102,53627.380431,53800.060045,53730.629102,53627.380431
4,2026-05-01,55387.344558,55308.448476,55191.069206,55387.344558,55308.448476,55191.069206


In [ ]:
# choose MW Required for Data Center
#Small -20MW, Med -30MW, Large Hyperscale -50 MW
#Assuming MW per DC =30MW
MW_PER_DC = 30


In [ ]:
#Calculating Number of Data Center per Month

import numpy as np

dc_capacity["dc_low"]  = np.floor(dc_capacity["surplus_low"]  / MW_PER_DC)
dc_capacity["dc_med"]  = np.floor(dc_capacity["surplus_med"]  / MW_PER_DC)
dc_capacity["dc_high"] = np.floor(dc_capacity["surplus_high"] / MW_PER_DC)
dc_capacity.head()


,ds,gap_low_mw,gap_med_mw,gap_high_mw,surplus_low,surplus_med,surplus_high,dc_low,dc_med,dc_high
0,2026-01-01,64421.494953,64419.057341,64415.445223,64421.494953,64419.057341,64415.445223,2147.0,2147.0,2147.0
1,2026-02-01,64398.681431,64352.981888,64285.137687,64398.681431,64352.981888,64285.137687,2146.0,2145.0,2142.0
2,2026-03-01,58587.261700,58531.974732,58449.836552,58587.261700,58531.974732,58449.836552,1952.0,1951.0,1948.0
3,2026-04-01,53800.060045,53730.629102,53627.380431,53800.060045,53730.629102,53627.380431,1793.0,1791.0,1787.0
4,2026-05-01,55387.344558,55308.448476,55191.069206,55387.344558,55308.448476,55191.069206,1846.0,1843.0,1839.0


In [ ]:

# Convert PJME DC capacity - NJ (PSEG) capacity using peak ratio


import pandas as pd

# PJME peak by year (from hourly PJME load)
load_tmp = load.copy()
load_tmp["year"] = load_tmp["dt"].dt.year
pjme_peak_by_year = load_tmp.groupby("year")["load_mw"].max().reset_index(name="pjme_peak_mw")

print("PJME peak by year:")
print(pjme_peak_by_year)

#  PSEG peak by year (from annual peak file)
pseg_peak_by_year = peak[["year", "nspl_mw"]].rename(columns={"nspl_mw": "pseg_peak_mw"}).copy()

print("\nPSEG peak by year:")
print(pseg_peak_by_year)

# Compute NJ share by year + average share
share = pd.merge(pseg_peak_by_year, pjme_peak_by_year, on="year", how="inner")
share["nj_share"] = share["pseg_peak_mw"] / share["pjme_peak_mw"]

print("\nNJ share by year:")
print(share)

avg_nj_share = share["nj_share"].mean()
print("\nAverage NJ share:", avg_nj_share)

# Add YEAR + NJ-scaled DC capacity into dc_capacity table

dc_capacity["ds"] = pd.to_datetime(dc_capacity["ds"], errors="coerce")
dc_capacity["year"] = dc_capacity["ds"].dt.year
dc_capacity["nj_dc_low"] =( dc_capacity["dc_low"] * avg_nj_share).astype(int)
dc_capacity["nj_dc_med"] =( dc_capacity["dc_med"] * avg_nj_share).astype(int)
dc_capacity["nj_dc_high"] =(dc_capacity["dc_high"] * avg_nj_share).astype(int)

print("\nPJME vs NJ-scaled DC capacity (first 5 rows):")
print(dc_capacity[["year", "ds", "dc_low", "nj_dc_low","dc_med", "nj_dc_med","dc_high","nj_dc_high"]].head())


PJME peak by year:
   year  pjme_peak_mw
0  2020         55961
1  2021         56509
2  2022         56427
3  2023         54508
4  2024         58036
5  2025         57965

PSEG peak by year:
   year  pseg_peak_mw
0  2026       10229.5
1  2025       10151.7
2  2024        9561.0
3  2023       10147.0
4  2022       10064.1

NJ share by year:
   year  pseg_peak_mw  pjme_peak_mw  nj_share
0  2025       10151.7         57965  0.175135
1  2024        9561.0         58036  0.164743
2  2023       10147.0         54508  0.186156
3  2022       10064.1         56427  0.178356

Average NJ share: 0.17609745887844452

PJME vs NJ-scaled DC capacity (first 5 rows):
   year         ds  dc_low  nj_dc_low  dc_med  nj_dc_med  dc_high  nj_dc_high
0  2026 2026-01-01  2147.0        378  2147.0        378   2147.0         378
1  2026 2026-02-01  2146.0        377  2145.0        377   2142.0         377
2  2026 2026-03-01  1952.0        343  1951.0        343   1948.0         343
3  2026 2026-04-01  1793.0  